# 🚪 Módulo 5 — Salida de Personal
**Sistema de Gestión de Recursos Humanos**

Este módulo gestiona dos tipos de salida:
- **Salida por Movida** → El empleado cambia de puesto y/o departamento dentro de la empresa. Sigue activo.
- **Salida Definitiva** → El empleado deja la empresa. Su estado pasa a `terminado`.

> **Base de datos:** `rrhh_sistema` · **Puerto:** `1989` · **Motor:** MySQL

In [ ]:
# ── 1. Instalación automática de dependencias ─────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'mysql-connector-python', 'ipywidgets', '--quiet'], check=False)

# ── Importaciones ─────────────────────────────────────────────────────────
import mysql.connector
from mysql.connector import Error
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
from datetime import date

print('✅ Librerías cargadas correctamente.')

In [ ]:
# Conexion a la base de datos
import os
import mysql.connector

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'port': int(os.getenv('DB_PORT', '1989')),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'rrhh_sistema'),
    'charset': 'utf8mb4',
}

def get_conn():
    return mysql.connector.connect(**DB_CONFIG)

try:
    conn = get_conn()
    conn.close()
    print(f'Conexion exitosa -> {DB_CONFIG["database"]} (Puerto {DB_CONFIG["port"]})')
except mysql.connector.Error as e:
    print(f'Error de conexion: {e}')



In [ ]:
# ── 3. Crear tabla historial_movimientos si no existe ─────────────────────
# Guarda la trazabilidad de cada salida por movida (traslado interno).

SQL_HISTORIAL = """
CREATE TABLE IF NOT EXISTS historial_movimientos (
    id_movimiento    INT          AUTO_INCREMENT PRIMARY KEY,
    codigo_empresa   CHAR(5)      NOT NULL,
    fecha_movimiento DATE         NOT NULL,
    puesto_anterior  VARCHAR(100) NOT NULL,
    puesto_nuevo     VARCHAR(100) NOT NULL,
    depto_anterior   INT          NOT NULL,
    depto_nuevo      INT          NOT NULL,
    motivo           VARCHAR(255) NULL,
    FOREIGN KEY (codigo_empresa) REFERENCES empleados(codigo_empresa) ON UPDATE CASCADE,
    FOREIGN KEY (depto_anterior) REFERENCES departamentos(id_departamento),
    FOREIGN KEY (depto_nuevo)    REFERENCES departamentos(id_departamento)
);
"""

try:
    conn = get_conn()
    cur  = conn.cursor()
    cur.execute(SQL_HISTORIAL)
    conn.commit()
    cur.close(); conn.close()
    print('✅ Tabla historial_movimientos lista.')
except Error as e:
    print(f'❌ Error al crear tabla: {e}')

In [ ]:
# ── 4. Funciones CRUD ─────────────────────────────────────────────────────

def get_departamentos() -> list:
    """Devuelve [(id, nombre), ...] de departamentos."""
    conn = get_conn()
    cur  = conn.cursor()
    cur.execute('SELECT id_departamento, nombre_departamento FROM departamentos ORDER BY nombre_departamento')
    rows = cur.fetchall()
    cur.close(); conn.close()
    return rows

def buscar_empleado(codigo: str) -> dict:
    """Retorna datos completos del empleado o None si no existe."""
    sql = """
        SELECT e.codigo_empresa, e.nombre, e.apellido, e.numero_cedula, e.estado,
               p.puesto, p.fecha_ingreso, p.id_departamento, d.nombre_departamento
        FROM   empleados e
        JOIN   personal      p ON p.codigo_empresa  = e.codigo_empresa
        JOIN   departamentos d ON d.id_departamento = p.id_departamento
        WHERE  e.codigo_empresa = %s
    """
    conn = get_conn()
    cur  = conn.cursor(dictionary=True)
    cur.execute(sql, (codigo.upper(),))
    row = cur.fetchone()
    cur.close(); conn.close()
    return row

# ──────────────────────────────────────────────────────────────────────────
def registrar_movida(codigo: str, nuevo_puesto: str, nuevo_depto_id: int, motivo: str) -> str:
    """
    Salida por Movida:
      - Inserta en historial_movimientos.
      - Actualiza puesto e id_departamento en personal.
      - El empleado continúa ACTIVO.
    """
    emp = buscar_empleado(codigo)
    if not emp:
        return f'❌ No se encontró el empleado con código {codigo}.'
    if emp['estado'] == 'terminado':
        return f'❌ {emp["nombre"]} {emp["apellido"]} ya tiene salida definitiva. No se puede mover.'
    conn = get_conn()
    cur  = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO historial_movimientos
                (codigo_empresa, fecha_movimiento, puesto_anterior, puesto_nuevo,
                 depto_anterior, depto_nuevo, motivo)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (codigo.upper(), date.today(), emp['puesto'], nuevo_puesto,
               emp['id_departamento'], nuevo_depto_id, motivo or None))
        cur.execute("""
            UPDATE personal SET puesto = %s, id_departamento = %s
            WHERE codigo_empresa = %s
        """, (nuevo_puesto, nuevo_depto_id, codigo.upper()))
        conn.commit()
        return f'✅ Movida registrada: {emp["nombre"]} {emp["apellido"]} → {nuevo_puesto}'
    except Error as e:
        conn.rollback()
        return f'❌ Error: {e}'
    finally:
        cur.close(); conn.close()

# ──────────────────────────────────────────────────────────────────────────
def registrar_salida_definitiva(codigo: str, fecha_salida, motivo: str, observaciones: str) -> str:
    """
    Salida Definitiva:
      - Inserta en salida_personal.
      - Cambia estado a 'terminado' en empleados.
    """
    emp = buscar_empleado(codigo)
    if not emp:
        return f'❌ No se encontró el empleado con código {codigo}.'
    if emp['estado'] == 'terminado':
        return f'⚠️ {emp["nombre"]} {emp["apellido"]} ya tiene salida definitiva registrada.'
    conn = get_conn()
    cur  = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO salida_personal (codigo_empresa, fecha_salida, motivo_salida, observaciones)
            VALUES (%s, %s, %s, %s)
        """, (codigo.upper(), fecha_salida, motivo, observaciones or None))
        cur.execute(
            "UPDATE empleados SET estado = 'terminado' WHERE codigo_empresa = %s",
            (codigo.upper(),)
        )
        conn.commit()
        return f'✅ Salida definitiva registrada: {emp["nombre"]} {emp["apellido"]}'
    except mysql.connector.IntegrityError:
        conn.rollback()
        return f'⚠️ Ya existe una salida definitiva para el código {codigo}.'
    except Error as e:
        conn.rollback()
        return f'❌ Error: {e}'
    finally:
        cur.close(); conn.close()

print('✅ Funciones CRUD cargadas.')

In [ ]:
# ── 5. PANEL PRINCIPAL CON PESTAÑAS ──────────────────────────────────────

STYLE  = {'description_width': '160px'}
LAYOUT = widgets.Layout(width='420px')
BTN_LY = widgets.Layout(width='200px', margin='8px 4px 0 0')
OUT_LY = widgets.Layout(margin='8px 0 0 0', border='1px solid #e0e0e0',
                         padding='10px', border_radius='6px', min_height='40px')

# Cargar departamentos para los dropdowns
deptos_raw = get_departamentos()
depto_opts = {nombre: id_ for id_, nombre in deptos_raw}   # {'Ventas': 1, ...}


# ══════════════════════════════════════════════════════════════════════════
# TAB 1 — VER TODOS LOS EMPLEADOS
# ══════════════════════════════════════════════════════════════════════════
btn_ver = widgets.Button(description='🔄 Actualizar lista',
                          button_style='info', layout=BTN_LY)
out_ver = widgets.Output(layout=OUT_LY)

def on_ver(_):
    with out_ver:
        clear_output(wait=True)
        try:
            conn = get_conn()
            df = pd.read_sql("""
                SELECT e.codigo_empresa AS Código,
                       e.nombre         AS Nombre,
                       e.apellido       AS Apellido,
                       p.puesto         AS Puesto,
                       d.nombre_departamento AS Departamento,
                       p.fecha_ingreso  AS Ingreso,
                       e.estado         AS Estado
                FROM   empleados e
                JOIN   personal      p ON p.codigo_empresa  = e.codigo_empresa
                JOIN   departamentos d ON d.id_departamento = p.id_departamento
                ORDER  BY e.apellido, e.nombre
            """, conn)
            conn.close()
            if df.empty:
                print('(No hay empleados registrados.)')
            else:
                display(df.style.set_table_styles([
                    {'selector': 'th', 'props': [('background', '#2c3e50'), ('color', 'white'), ('padding', '6px 12px')]},
                    {'selector': 'td', 'props': [('padding', '5px 12px')]}
                ]).hide(axis='index'))
        except Error as e:
            print(f'❌ {e}')

btn_ver.on_click(on_ver)
tab1 = widgets.VBox([
    widgets.HTML('<b>Lista de todos los empleados</b>'),
    btn_ver, out_ver
])


# ══════════════════════════════════════════════════════════════════════════
# TAB 2 — BUSCAR EMPLEADO
# ══════════════════════════════════════════════════════════════════════════
txt_buscar = widgets.Text(description='Código empresa:', style=STYLE, layout=LAYOUT,
                           placeholder='Ej: GLIAE')
btn_buscar = widgets.Button(description='🔍 Buscar',
                             button_style='warning', layout=BTN_LY)
out_buscar = widgets.Output(layout=OUT_LY)

def on_buscar(_):
    with out_buscar:
        clear_output(wait=True)
        codigo = txt_buscar.value.strip().upper()
        if not codigo:
            print('⚠️ Ingresa un código de empleado.')
            return
        emp = buscar_empleado(codigo)
        if not emp:
            print(f'❌ No se encontró ningún empleado con código "{codigo}".')
            return
        color = '#2e7d32' if emp['estado'] == 'activo' else '#c62828'
        display(HTML(f"""
        <div style='background:#f5f5f5;border:1px solid #ccc;border-radius:8px;padding:14px'>
            <table style='border-collapse:collapse;width:100%'>
                <tr><td style='padding:4px 14px 4px 0'><b>Código:</b></td><td><code>{emp['codigo_empresa']}</code></td></tr>
                <tr><td><b>Nombre:</b></td><td>{emp['nombre']} {emp['apellido']}</td></tr>
                <tr><td><b>Cédula:</b></td><td>{emp['numero_cedula']}</td></tr>
                <tr><td><b>Puesto actual:</b></td><td>{emp['puesto']}</td></tr>
                <tr><td><b>Departamento:</b></td><td>{emp['nombre_departamento']}</td></tr>
                <tr><td><b>Fecha ingreso:</b></td><td>{emp['fecha_ingreso']}</td></tr>
                <tr><td><b>Estado:</b></td><td><b style='color:{color}'>{emp['estado'].upper()}</b></td></tr>
            </table>
        </div>
        """))

btn_buscar.on_click(on_buscar)
tab2 = widgets.VBox([
    widgets.HTML('<b>Buscar empleado por código de empresa</b>'),
    txt_buscar, btn_buscar, out_buscar
])


# ══════════════════════════════════════════════════════════════════════════
# TAB 3 — SALIDA POR MOVIDA
# ══════════════════════════════════════════════════════════════════════════
txt_mov_codigo = widgets.Text(description='Código empleado:', style=STYLE, layout=LAYOUT,
                               placeholder='Ej: GLIAE')
txt_mov_puesto = widgets.Text(description='Nuevo puesto:', style=STYLE, layout=LAYOUT,
                               placeholder='Ej: Coordinador de Logística')
drp_mov_depto  = widgets.Dropdown(description='Nuevo departamento:', style=STYLE, layout=LAYOUT,
                                   options=list(depto_opts.keys()) if depto_opts else ['(Sin departamentos)'])
txt_mov_motivo = widgets.Text(description='Motivo traslado:', style=STYLE, layout=LAYOUT,
                               placeholder='Ej: Ascenso, reorganización...')
btn_mov_ver    = widgets.Button(description='🔍 Verificar empleado',
                                 button_style='warning', layout=BTN_LY)
btn_mov_guard  = widgets.Button(description='💾 Registrar movida',
                                 button_style='primary', layout=BTN_LY)
out_mov        = widgets.Output(layout=OUT_LY)

def on_mov_ver(_):
    with out_mov:
        clear_output(wait=True)
        codigo = txt_mov_codigo.value.strip().upper()
        if not codigo:
            print('⚠️ Ingresa el código del empleado.')
            return
        emp = buscar_empleado(codigo)
        if not emp:
            print(f'❌ No se encontró el empleado "{codigo}".')
            return
        if emp['estado'] == 'terminado':
            print(f'❌ {emp["nombre"]} {emp["apellido"]} ya tiene salida definitiva. No se puede mover.')
            return
        print(f'📌 Empleado encontrado: {emp["nombre"]} {emp["apellido"]}')
        print(f'   Puesto actual    : {emp["puesto"]}')
        print(f'   Departamento     : {emp["nombre_departamento"]}')
        print('   ↓ Completa los campos y presiona Registrar movida.')

def on_mov_guard(_):
    with out_mov:
        clear_output(wait=True)
        codigo       = txt_mov_codigo.value.strip().upper()
        nuevo_puesto = txt_mov_puesto.value.strip()
        if not codigo:
            print('⚠️ Ingresa el código del empleado.')
            return
        if not nuevo_puesto:
            print('⚠️ El nuevo puesto no puede estar vacío.')
            return
        if not depto_opts:
            print('❌ No hay departamentos en la base de datos. Ejecuta el Módulo 0 primero.')
            return
        nuevo_depto_id = depto_opts.get(drp_mov_depto.value)
        msg = registrar_movida(codigo, nuevo_puesto, nuevo_depto_id, txt_mov_motivo.value.strip())
        print(msg)
        if msg.startswith('✅'):
            txt_mov_puesto.value = ''
            txt_mov_motivo.value = ''

btn_mov_ver.on_click(on_mov_ver)
btn_mov_guard.on_click(on_mov_guard)
tab3 = widgets.VBox([
    widgets.HTML('<b>Salida por Movida</b><br>'
                 '<small style="color:gray">El empleado cambia de puesto o departamento. '
                 'Sigue <b>activo</b> en la empresa.</small>'),
    txt_mov_codigo,
    widgets.HBox([btn_mov_ver, widgets.Label('← verifica primero el código')]),
    txt_mov_puesto,
    drp_mov_depto,
    txt_mov_motivo,
    btn_mov_guard,
    out_mov
])


# ══════════════════════════════════════════════════════════════════════════
# TAB 4 — SALIDA DEFINITIVA
# ══════════════════════════════════════════════════════════════════════════
MOTIVOS = [
    'Renuncia voluntaria',
    'Despido justificado',
    'Despido injustificado',
    'Jubilación',
    'Fin de contrato',
    'Mutuo acuerdo',
    'Fallecimiento',
    'Otro'
]

txt_def_codigo = widgets.Text(description='Código empleado:', style=STYLE, layout=LAYOUT,
                               placeholder='Ej: GLIAE')
dat_def_fecha  = widgets.DatePicker(description='Fecha de salida:', style=STYLE,
                                     layout=LAYOUT, value=date.today())
drp_def_motivo = widgets.Dropdown(description='Motivo de salida:', style=STYLE,
                                   layout=LAYOUT, options=MOTIVOS)
txt_def_obs    = widgets.Textarea(description='Observaciones:', style=STYLE,
                                   layout=widgets.Layout(width='420px', height='80px'),
                                   placeholder='Notas adicionales (opcional)')
chk_def_conf   = widgets.Checkbox(
                    description='Confirmo que deseo dar de baja definitivamente a este empleado',
                    value=False, layout=widgets.Layout(width='480px'))
btn_def_ver    = widgets.Button(description='🔍 Verificar empleado',
                                 button_style='warning', layout=BTN_LY)
btn_def_guard  = widgets.Button(description='🚪 Registrar baja definitiva',
                                 button_style='danger', layout=BTN_LY)
out_def        = widgets.Output(layout=OUT_LY)

def on_def_ver(_):
    with out_def:
        clear_output(wait=True)
        codigo = txt_def_codigo.value.strip().upper()
        if not codigo:
            print('⚠️ Ingresa el código del empleado.')
            return
        emp = buscar_empleado(codigo)
        if not emp:
            print(f'❌ No se encontró el empleado "{codigo}".')
            return
        color = '#2e7d32' if emp['estado'] == 'activo' else '#c62828'
        display(HTML(f"""
        <div style='background:#fff3e0;border:1px solid #fb8c00;border-radius:8px;padding:14px'>
            <b>⚠️ Revisa bien los datos antes de confirmar la baja:</b>
            <table style='margin-top:8px;border-collapse:collapse'>
                <tr><td style='padding:3px 14px 3px 0'><b>Código:</b></td><td><code>{emp['codigo_empresa']}</code></td></tr>
                <tr><td><b>Nombre:</b></td><td>{emp['nombre']} {emp['apellido']}</td></tr>
                <tr><td><b>Puesto:</b></td><td>{emp['puesto']}</td></tr>
                <tr><td><b>Departamento:</b></td><td>{emp['nombre_departamento']}</td></tr>
                <tr><td><b>Fecha ingreso:</b></td><td>{emp['fecha_ingreso']}</td></tr>
                <tr><td><b>Estado actual:</b></td><td><b style='color:{color}'>{emp['estado'].upper()}</b></td></tr>
            </table>
        </div>
        """))

def on_def_guard(_):
    with out_def:
        clear_output(wait=True)
        codigo = txt_def_codigo.value.strip().upper()
        if not codigo:
            print('⚠️ Ingresa el código del empleado.')
            return
        if not chk_def_conf.value:
            print('⚠️ Marca la casilla de confirmación antes de registrar la baja.')
            return
        if not dat_def_fecha.value:
            print('⚠️ Selecciona la fecha de salida.')
            return
        msg = registrar_salida_definitiva(
            codigo,
            dat_def_fecha.value,
            drp_def_motivo.value,
            txt_def_obs.value.strip()
        )
        if msg.startswith('✅'):
            emp = buscar_empleado(codigo)
            nombre = f"{emp['nombre']} {emp['apellido']}" if emp else codigo
            display(HTML(f"""
            <div style='background:#ffebee;border:1px solid #ef9a9a;border-radius:8px;padding:16px'>
                <h4 style='color:#c62828;margin:0'>🚪 Baja definitiva registrada</h4>
                <table style='margin-top:10px;border-collapse:collapse'>
                    <tr><td style='padding:4px 14px 4px 0'><b>Empleado:</b></td><td>{nombre}</td></tr>
                    <tr><td><b>Código:</b></td><td><code>{codigo}</code></td></tr>
                    <tr><td><b>Fecha de salida:</b></td><td>{dat_def_fecha.value}</td></tr>
                    <tr><td><b>Motivo:</b></td><td>{drp_def_motivo.value}</td></tr>
                    <tr><td><b>Estado:</b></td><td><b style='color:#c62828'>TERMINADO</b></td></tr>
                </table>
            </div>
            """))
            txt_def_codigo.value = ''
            txt_def_obs.value    = ''
            chk_def_conf.value   = False
        else:
            print(msg)

btn_def_ver.on_click(on_def_ver)
btn_def_guard.on_click(on_def_guard)
tab4 = widgets.VBox([
    widgets.HTML('<b>Salida Definitiva</b><br>'
                 '<small style="color:gray">⚠️ El empleado saldrá permanentemente de la empresa. '
                 'Su estado cambiará a <b style="color:#c62828">terminado</b>.</small>'),
    txt_def_codigo,
    widgets.HBox([btn_def_ver, widgets.Label('← verifica primero el código')]),
    dat_def_fecha,
    drp_def_motivo,
    txt_def_obs,
    chk_def_conf,
    btn_def_guard,
    out_def
])


# ══════════════════════════════════════════════════════════════════════════
# TAB 5 — HISTORIAL DE MOVIMIENTOS INTERNOS
# ══════════════════════════════════════════════════════════════════════════
btn_hist = widgets.Button(description='🔄 Ver historial',
                           button_style='info', layout=BTN_LY)
out_hist = widgets.Output(layout=OUT_LY)

def on_hist(_):
    with out_hist:
        clear_output(wait=True)
        try:
            conn = get_conn()
            df = pd.read_sql("""
                SELECT h.fecha_movimiento          AS Fecha,
                       h.codigo_empresa             AS Código,
                       CONCAT(e.nombre,' ',e.apellido) AS Empleado,
                       h.puesto_anterior            AS 'Puesto anterior',
                       h.puesto_nuevo               AS 'Puesto nuevo',
                       d1.nombre_departamento       AS 'Depto. anterior',
                       d2.nombre_departamento       AS 'Depto. nuevo',
                       COALESCE(h.motivo, '—')      AS Motivo
                FROM   historial_movimientos h
                JOIN   empleados     e  ON h.codigo_empresa = e.codigo_empresa
                JOIN   departamentos d1 ON h.depto_anterior = d1.id_departamento
                JOIN   departamentos d2 ON h.depto_nuevo    = d2.id_departamento
                ORDER  BY h.fecha_movimiento DESC
            """, conn)
            conn.close()
            if df.empty:
                print('(No hay movimientos internos registrados aún.)')
            else:
                display(df.style.set_table_styles([
                    {'selector': 'th', 'props': [('background', '#1565c0'), ('color', 'white'), ('padding', '6px 12px')]},
                    {'selector': 'td', 'props': [('padding', '5px 12px')]}
                ]).hide(axis='index'))
        except Error as e:
            print(f'❌ {e}')

btn_hist.on_click(on_hist)
tab5 = widgets.VBox([
    widgets.HTML('<b>Historial de Movimientos Internos (Salidas por Movida)</b>'),
    btn_hist, out_hist
])


# ══════════════════════════════════════════════════════════════════════════
# TAB 6 — REPORTE DE SALIDAS DEFINITIVAS
# ══════════════════════════════════════════════════════════════════════════
btn_rep = widgets.Button(description='🔄 Ver salidas definitivas',
                          button_style='info', layout=BTN_LY)
out_rep = widgets.Output(layout=OUT_LY)

def on_rep(_):
    with out_rep:
        clear_output(wait=True)
        try:
            conn = get_conn()
            df = pd.read_sql("""
                SELECT s.fecha_salida                   AS 'Fecha salida',
                       s.codigo_empresa                 AS Código,
                       CONCAT(e.nombre,' ',e.apellido)  AS Empleado,
                       p.puesto                         AS 'Último puesto',
                       d.nombre_departamento             AS Departamento,
                       p.fecha_ingreso                  AS Ingreso,
                       DATEDIFF(s.fecha_salida, p.fecha_ingreso) AS 'Días laborados',
                       s.motivo_salida                  AS Motivo,
                       COALESCE(s.observaciones, '—')   AS Observaciones
                FROM   salida_personal s
                JOIN   empleados     e ON s.codigo_empresa  = e.codigo_empresa
                JOIN   personal      p ON s.codigo_empresa  = p.codigo_empresa
                JOIN   departamentos d ON p.id_departamento = d.id_departamento
                ORDER  BY s.fecha_salida DESC
            """, conn)
            conn.close()
            if df.empty:
                print('(No hay salidas definitivas registradas aún.)')
            else:
                display(df.style.set_table_styles([
                    {'selector': 'th', 'props': [('background', '#b71c1c'), ('color', 'white'), ('padding', '6px 12px')]},
                    {'selector': 'td', 'props': [('padding', '5px 12px')]}
                ]).hide(axis='index'))
        except Error as e:
            print(f'❌ {e}')

btn_rep.on_click(on_rep)
tab6 = widgets.VBox([
    widgets.HTML('<b>Reporte de Salidas Definitivas</b>'),
    btn_rep, out_rep
])


# ══════════════════════════════════════════════════════════════════════════
# PANEL PRINCIPAL — PESTAÑAS
# ══════════════════════════════════════════════════════════════════════════
tabs = widgets.Tab(children=[tab1, tab2, tab3, tab4, tab5, tab6])
tabs.set_title(0, '👥 Empleados')
tabs.set_title(1, '🔍 Buscar')
tabs.set_title(2, '🔄 Salida por Movida')
tabs.set_title(3, '🚪 Salida Definitiva')
tabs.set_title(4, '📦 Historial Movidas')
tabs.set_title(5, '📋 Reporte Bajas')

header = widgets.HTML(
    '<div style="background:#2c3e50;color:white;padding:12px 16px;'
    'border-radius:6px 6px 0 0;font-family:Arial;font-size:15px;">'
    '🚪 <b>Módulo 5 — Salida de Personal</b> | '
    'Sistema RRHH · rrhh_sistema</div>'
)

panel = widgets.VBox([header, tabs])
display(panel)

# Carga automática al abrir el panel
on_ver(None)